# Tracking Hyperparameter Tuning with MLflow

Hyperparameter tuning is an important process for improving the performance of a machine learning model.  
Manually tracking and comparing the different trials is cumbersome — MLflow provides a framework that makes it systematic.

By the end of this tutorial you will know how to:

- Set up your environment with MLflow tracking.  
- Define an **objective function** that fits a model and can be plugged into a hyperparameter tuning library.  
- Use [Optuna](https://optuna.org/) for hyperparameter tuning.  
- Leverage **child runs** within MLflow to keep one row per trial under a single parent run.

This is a modification of the official MLflow tutorial: **[Hyperparameter Tuning with MLflow and Optuna](https://mlflow.org/docs/latest/ml/getting-started/hyperparameter-tuning/)**.  
Where this notebook corrects, supplements, or deliberately diverges from upstream, the relationship is named with an inline bold callout — one of **Bug**, **Stale**, **Missing from**, or **Diverges from upstream tutorial:** — or, in code cells, with a `NOTE (differs from upstream)` comment.  
Sections with no upstream counterpart use a plain heading and no callout.

## Prerequisites

**Diverges from upstream tutorial:**  
Upstream says `pip install mlflow optuna`. This repo uses **`uv` exclusively** for dependency management — `pip` is explicitly forbidden (see `CLAUDE.md`).  
`mlflow` and `scikit-learn` are already in `pyproject.toml`; you only need to add `optuna`:

```bash
uv add optuna
```

This updates `pyproject.toml`, regenerates `uv.lock`, and syncs `.venv/`. Commit both files together.

### Start the tracking server

**Missing from upstream tutorial:**  
Before running any cell that calls `mlflow.set_tracking_uri(...)`, start the MLflow server **in a separate terminal**, from `src/`:

```bash
cd src/
mlflow ui --host 127.0.0.1 --port 5000   # use 5001, 5002, … if 5000 is taken
```

Starting from `src/` keeps per-developer runtime state (`mlflow.db`, `mlartifacts/`) inside the source tree rather than scattering it at the repo root — same convention as `b_tracking_quickstart.ipynb`.  
If you change the port, update `PORT` in the `mlflow.set_tracking_uri(...)` cell below to match.

## A one-paragraph Optuna primer

The upstream tutorial assumes you already know Optuna. If you don't, here is the minimum vocabulary:

| Optuna concept | What it is |
|---|---|
| **Study** | The full optimization task — one search over one hyperparameter space. |
| **Trial** | A single evaluation: one combination of hyperparameters → one objective value. |
| **Objective function** | A user-defined function `objective(trial) -> float` that Optuna calls once per trial. Inside, you read suggested values via `trial.suggest_*`, train a model, and return the metric to minimize (or maximize). |
| **Sampler** | The algorithm that picks the next trial's hyperparameters. Optuna's default is **TPE** (Tree-structured Parzen Estimator), a Bayesian-style sampler that models *good* vs *bad* past trials and biases new suggestions toward promising regions. |
| **Direction** | `"minimize"` (e.g. for loss/MSE) or `"maximize"` (e.g. for accuracy/AUC). |

This pairs naturally with MLflow:

- each Optuna **trial** becomes one MLflow **child run**, and  
- the **study** becomes one MLflow **parent run** that groups them.

### Why TPE instead of grid search?

With 3 hyperparameters and ~10 candidate values each you already have 1000 fits to evaluate.  
TPE typically finds a near-optimal point in 20–50 trials by **learning from earlier results** instead of treating every point independently.

## Step 1: Create a new experiment

`mlflow.set_experiment(name)` creates the experiment if it doesn't exist, then makes it the active experiment for subsequent `start_run()` calls.

Before that, point the client at the locally running tracking server.

In [6]:
import mlflow

HOST = "127.0.0.1"
PORT = 5001
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

mlflow.set_experiment("Hyperparameter Tuning Experiment")

<Experiment: artifact_location='mlflow-artifacts:/3', creation_time=1779437471181, experiment_id='3', last_update_time=1779437471181, lifecycle_stage='active', name='Hyperparameter Tuning Experiment', tags={}, trace_location=None, workspace='default'>

## Step 2: Prepare the data

We use the California housing regression dataset. The task is to predict median house value from 8 numeric features (median income, house age, average rooms, etc.) — small enough that a `RandomForestRegressor` trains in seconds per trial, which keeps the 30-trial study below a minute on a laptop.

In [7]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

X, y = fetch_california_housing(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

## Step 3: Define the objective function

In Optuna a **study** is the full optimization session, made of many **trials**.   
In MLflow we mirror this with a **parent run** that contains many **child runs** (one per trial).  
The parent/child relationship is what gives you the grouped, filterable, comparable view in the MLflow UI later.

### Why bother with parent/child runs?

Without the parent/child grouping, a 10-trial sweep produces 10 flat sibling runs in your experiment, mixed in with any unrelated runs from other notebooks.  
Finding "the runs from yesterday's tuning study" then means scrolling, filtering by tag conventions you remembered to set, or sorting by timestamp and hoping you guessed the right window.

With nesting:

- **The parent** is one row in the runs table — the stable handle for the whole sweep, with the best parameters logged on it.  
- **The children** collapse under the parent — you only see them when you expand the row, but they're still individually filterable and comparable in chart view.

That structure is what makes MLflow's UI useful for sweeps rather than just for single runs.

### What the objective function does

The objective function below:

1. **Opens a nested MLflow run** (`nested=True`) — this is what makes it a child of whichever run is currently active.  
2. **Asks Optuna for suggested values** via `trial.suggest_int` / `trial.suggest_float`.  
   Each call returns one value in the given range that Optuna's sampler picks for this trial — biased toward regions that did well in earlier trials, with an optional `step` for ints (e.g. `step=10` → 50, 60, …, 300).  
3. **Fits a `RandomForestRegressor`**, scores it on the validation set, logs everything to MLflow.  
4. **Stores the MLflow `run_id` on the trial** via `trial.set_user_attr` so we can look up the *best* run later without having to query MLflow.

In [8]:
import optuna
import sklearn.ensemble
import sklearn.metrics


def objective(trial):
    # nested=True makes this run a child of the currently-active (parent) run.
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:
        params = {
            "max_depth": trial.suggest_int("rf_max_depth", 2, 32),
            "n_estimators": trial.suggest_int("rf_n_estimators", 50, 300, step=10),
            "max_features": trial.suggest_float("rf_max_features", 0.2, 1.0),
        }
        mlflow.log_params(params)

        regressor = sklearn.ensemble.RandomForestRegressor(**params, random_state=0)
        regressor.fit(X_train, y_train)

        y_pred = regressor.predict(X_val)
        error = sklearn.metrics.mean_squared_error(y_val, y_pred)
        mlflow.log_metric("error", error)

        # NOTE (differs from upstream tutorial): use the safer `skops` format
        # instead of the default cloudpickle. See the final markdown cell of
        # `tracking_quickstart.ipynb` for the full rationale — TL;DR: pickle/
        # cloudpickle execute arbitrary code on load, skops does not.
        mlflow.sklearn.log_model(
            regressor,
            name="model",
            serialization_format="skops",
        )

        # Stash the run_id on the trial so we can look up the best child run
        # later via `study.best_trial.user_attrs["run_id"]`.
        trial.set_user_attr("run_id", child_run.info.run_id)
        return error

## Step 4: Run the hyperparameter tuning study

*(The upstream tutorial labels this "Step 3" twice — fixing the numbering here.)*

We create a parent run named `"study"` and call `study.optimize(objective, n_trials=...)`.  
Because the objective opens runs with `nested=True`, each trial becomes a child run grouped under this parent.

After the study finishes we also log the best trial's parameters and metric onto the **parent** run.  
This makes the parent useful at a glance: you can see the winning hyperparameters without drilling into the child runs.

In [9]:
n_trials = 5

with mlflow.start_run(run_name="study") as parent_run:
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    # Promote the best trial's info onto the parent run for easy scanning.
    mlflow.log_params(study.best_trial.params)
    mlflow.log_metric("best_error", study.best_value)
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

print(f"Best trial: #{study.best_trial.number}")
print(f"Best error (MSE): {study.best_value:.4f}")
print(f"Best params: {study.best_trial.params}")
print(f"Best child run_id: {study.best_trial.user_attrs.get('run_id')}")

[I 2026-05-22 11:37:30,534] A new study created in memory with name: no-name-d56d142e-0f94-410f-9e28-9c6d3bcfead3
2026/05/22 11:37:46 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-05-22 11:37:47,366] Trial 0 finished with value: 0.2558869914957442 and parameters: {'rf_max_depth': 23, 'rf_n_estimators': 240, 'rf_max_features': 0.6785375833982914}. Best is trial 0 with value: 0.2558869914957442.


🏃 View run trial_0 at: http://127.0.0.1:5001/#/experiments/3/runs/cafa77cbd5354ef68d3f8d2f73dea3f3
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/22 11:37:56 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-05-22 11:37:57,222] Trial 1 finished with value: 0.3184972939626479 and parameters: {'rf_max_depth': 9, 'rf_n_estimators': 210, 'rf_max_features': 0.6817942101895889}. Best is trial 0 with value: 0.2558869914957442.


🏃 View run trial_1 at: http://127.0.0.1:5001/#/experiments/3/runs/ab89ab2656174843b9fd927722176cab
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/22 11:38:13 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-05-22 11:38:14,909] Trial 2 finished with value: 0.25155391883777617 and parameters: {'rf_max_depth': 30, 'rf_n_estimators': 290, 'rf_max_features': 0.599199324445102}. Best is trial 2 with value: 0.25155391883777617.


🏃 View run trial_2 at: http://127.0.0.1:5001/#/experiments/3/runs/6797983ace114a9a8153e65b9a4fc3a4
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/22 11:38:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-05-22 11:38:21,191] Trial 3 finished with value: 0.7369942865095334 and parameters: {'rf_max_depth': 2, 'rf_n_estimators': 280, 'rf_max_features': 0.5950686640263825}. Best is trial 2 with value: 0.25155391883777617.


🏃 View run trial_3 at: http://127.0.0.1:5001/#/experiments/3/runs/ad04df5c652f49c3be6bb8d6be241844
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3


2026/05/22 11:38:39 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
[I 2026-05-22 11:38:41,012] Trial 4 finished with value: 0.25710802399927796 and parameters: {'rf_max_depth': 20, 'rf_n_estimators': 300, 'rf_max_features': 0.7321538324605117}. Best is trial 2 with value: 0.25155391883777617.


🏃 View run trial_4 at: http://127.0.0.1:5001/#/experiments/3/runs/4f8bb1949619426fa462a7e7535a615a
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
🏃 View run study at: http://127.0.0.1:5001/#/experiments/3/runs/50dd33587835447b877bd2ece915dc25
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/3
Best trial: #2
Best error (MSE): 0.2516
Best params: {'rf_max_depth': 30, 'rf_n_estimators': 290, 'rf_max_features': 0.599199324445102}
Best child run_id: 6797983ace114a9a8153e65b9a4fc3a4


## Why bother with `trial.set_user_attr`?

The line

```python
trial.set_user_attr("run_id", child_run.info.run_id)
```

looks redundant — MLflow already knows the run id, so why store it on the trial?

The reason is that **Optuna's `study.best_trial` is the cheap, in-memory source of truth** for which configuration won, but it has no awareness of MLflow.  
Without `set_user_attr` you'd have to:

1. Pull all child runs of the parent from MLflow (`mlflow.search_runs(...)`),  
2. Sort them by the `error` metric,  
3. Hope nothing else logged to that parent in the meantime.

Stashing the `run_id` on the trial makes the lookup a single dict access (`study.best_trial.user_attrs["run_id"]`).  
Both the *configuration* and *where the model lives* travel together — that's what makes Step 6 below a one-liner.

### A note on the parent/child UI grouping

In the MLflow UI, child runs do **not** show up on the same flat list as the parent by default — they're hidden behind an expand-arrow on the parent row.

If you sort the experiment by metric and don't see your trials, you almost certainly:

- forgot `nested=True` (so the trial runs ended up as siblings of the parent, not children), or  
- are filtering them out.

**The fluent-API rule:** any `start_run` opened while another run is active becomes a child *only if* `nested=True`; otherwise MLflow raises.

## Step 5: View the results in the MLflow UI

Open the tracking UI at <http://127.0.0.1:5000/> (or whichever port you started `mlflow ui` on).  
Click into **Hyperparameter Tuning Experiment** and you should see:

- **One parent run** named `study` with `n_trials=5`, `best_error=...`, and the winning hyperparameters logged on it.  
- **5 child runs** (`trial_0` … `trial_4`) collapsed under the parent.

The two most useful UI features for tuning workflows:

1. **Chart view** (chart icon, top-left of the runs table) — plots `error` against any logged hyperparameter.  
   Use this to see *which* hyperparameters actually moved the metric and which were noise.  
   For a random forest on this dataset, `max_depth` and `n_estimators` typically dominate.  
2. **Parallel coordinates** (also under the chart view) — one line per trial across all hyperparameters, colored by `error`.  
   The best trials cluster, and the cluster's coordinates tell you where a tighter follow-up search should live.

If the UI shows nothing, double-check that the port in `mlflow.set_tracking_uri(...)` matches the port `mlflow ui` is listening on.

## Step 6: Register the best model

Once you have a winning trial you can register its model into the [MLflow Model Registry](https://mlflow.org/docs/latest/ml/model-registry.md) so it can be loaded by name (and version, or alias like `champion`) from anywhere that talks to the same tracking server.

**Bug in upstream tutorial:** the upstream tutorial registers the best model with:

```python
mlflow.register_model(
    model_uri="runs:/d0210c58afff4737a306a2fbc5f1ff8d/model",
    name="housing-price-predictor",
)
```

That run id is a placeholder from whoever wrote the docs — it won't exist on **your** tracking server, so copy-pasting that cell errors out with `Run not found`. The fix is to use the `run_id` we stashed on the best trial in Step 3.

In [10]:
best_run_id = study.best_trial.user_attrs["run_id"]

registered = mlflow.register_model(
    model_uri=f"runs:/{best_run_id}/model",
    name="housing-price-predictor",
)

print(f"Registered '{registered.name}' as version {registered.version}")

Registered model 'housing-price-predictor' already exists. Creating a new version of this model...
2026/05/22 11:38:41 WARNING mlflow.tracking._model_registry.fluent: Run with id 6797983ace114a9a8153e65b9a4fc3a4 has no artifacts at artifact path 'model', registering model based on models:/m-fcce1b1a68144c28a677d25bf4b22029 instead
2026/05/22 11:38:41 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: housing-price-predictor, version 3
Created version '3' of model 'housing-price-predictor'.


Registered 'housing-price-predictor' as version 3


Re-running the cell registers a *new version* of the same model — the registry never overwrites, it appends. You can promote a specific version with an alias:

```python
from mlflow import MlflowClient
MlflowClient().set_registered_model_alias(
    name="housing-price-predictor",
    alias="champion",
    version=registered.version,
)
```

and then load it by alias from production code:

```python
model = mlflow.pyfunc.load_model("models:/housing-price-predictor@champion")
```

This is the pattern that decouples "which artifact is in production" from "which run produced it" — exactly what the registry is for.

## Next steps

- [MLflow Tracking](https://mlflow.org/docs/latest/ml/tracking.md) — the full tracking API.  
- [MLflow Model Registry](https://mlflow.org/docs/latest/ml/model-registry.md) — versions, aliases, and lifecycle management.  
- [Optuna documentation](https://optuna.readthedocs.io/) — more samplers (CMA-ES, NSGA-II), pruners (early-stopping unpromising trials), and multi-objective optimization.  
- [MLflow for Deep Learning](https://mlflow.org/docs/latest/ml/deep-learning.md) — the same parent/child pattern, but with PyTorch/TensorFlow flavors and per-epoch metric logging.